# 🇮🇳 Indian Ocean & Coastal AIS Trajectory Generator

This notebook generates high-fidelity **AIS mock trajectory datasets** along official Indian coastal shipping corridors:
- **Arabian Sea & West Coast Corridor**: Gulf of Kutch → Saurashtra → Mumbai High Oil Fields → JNPT Port → Mormugao (Goa) → Kochi Container Terminal.
- **Bay of Bengal & East Coast Corridor**: Haldia / Kolkata Dock Complex → Dhamra → Paradip → Visakhapatnam → Krishna-Godavari Basin → Chennai Port.
- **South Indian Trans-Oceanic Highway**: 8-Degree Channel (Minicoy / Lakshadweep) → Cape Comorin → Sri Lanka South Coast transit.
- **Coast Guard Offshore Patrol**: Mumbai Sector 4 EEZ Intercept Grid.

## 1. Imports & Configuration

In [ ]:
import math
import random
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

OUT_CSV = Path('../../data/AIS_INDIAN_OCEAN_MOCK.csv')
print(f'Target Output CSV: {OUT_CSV}')

## 2. Define Indian Maritime Corridors & Vessels

In [ ]:
VESSEL_CORRIDORS = [
    {
        'mmsi': 419000123,
        'name': 'MT DESH SHANTI',
        'type': 80,
        'type_str': 'VLCC Crude Tanker',
        'length': 333, 'width': 60, 'draft': 21.5, 'cargo': 80,
        'base_speed': 13.8,
        'waypoints': [
            (22.45, 68.90),  # Gulf of Kutch entrance
            (21.20, 69.80),  # Saurashtra coast
            (19.80, 71.30),  # Approaching Mumbai High
            (19.10, 72.10),  # Mumbai High West
            (18.90, 72.75),  # Mumbai Anchorage Outer
            (18.95, 72.88),  # JNPT / Jawahar Dweep Crude Berth
        ],
        'start_time': '2026-03-01T04:00:00Z',
        'total_pings': 120,
    },
    {
        'mmsi': 419000456,
        'name': 'MV KAVERI PRIDE',
        'type': 70,
        'type_str': 'Ultra Large Container Vessel',
        'length': 366, 'width': 48, 'draft': 14.2, 'cargo': 71,
        'base_speed': 18.5,
        'waypoints': [
            (22.65, 69.60),  # Mundra Port
            (20.50, 71.00),  # Gujarat Offshore
            (18.95, 72.70),  # Mumbai Outer
            (16.50, 73.10),  # Ratnagiri Offshore
            (15.40, 73.60),  # Mormugao Offshore
            (12.80, 74.60),  # Mangalore Offshore
            (9.96, 76.22),   # Kochi Container Terminal (Vallarpadam)
        ],
        'start_time': '2026-03-01T01:30:00Z',
        'total_pings': 140,
    },
    {
        'mmsi': 419000789,
        'name': 'MT SAMUDRANIDHI',
        'type': 80,
        'type_str': 'Chemical / Product Tanker',
        'length': 183, 'width': 32, 'draft': 11.8, 'cargo': 82,
        'base_speed': 12.5,
        'waypoints': [
            (9.90, 76.15),   # Kochi Refinery Outer
            (8.50, 76.80),   # Vizhinjam Offshore
            (7.90, 77.55),   # Kanyakumari South Tip
            (8.75, 78.20),   # Tuticorin Port
            (10.50, 79.95),  # Nagapattinam Offshore
            (12.00, 80.20),  # Puducherry Offshore
            (13.10, 80.35),  # Chennai Port (Bharathi Dock)
        ],
        'start_time': '2026-03-01T06:00:00Z',
        'total_pings': 130,
    },
    {
        'mmsi': 419000234,
        'name': 'MV CHOLA TRADER',
        'type': 70,
        'type_str': 'Capesize Bulk Carrier',
        'length': 292, 'width': 45, 'draft': 17.5, 'cargo': 70,
        'base_speed': 11.2,
        'waypoints': [
            (20.25, 86.68),  # Paradip Port
            (19.00, 85.10),  # Gopalpur Offshore
            (17.65, 83.32),  # Visakhapatnam Port
            (16.90, 82.35),  # Kakinada Deepwater Port
            (15.50, 80.50),  # Machilipatnam Offshore
            (14.25, 80.12),  # Krishnapatnam Port
        ],
        'start_time': '2026-03-01T03:00:00Z',
        'total_pings': 110,
    },
    {
        'mmsi': 419000567,
        'name': 'MT RATNA SAGAR',
        'type': 80,
        'type_str': 'Crude Shuttle Tanker',
        'length': 245, 'width': 42, 'draft': 14.0, 'cargo': 80,
        'base_speed': 13.0,
        'waypoints': [
            (19.45, 71.35),  # Mumbai High North Platform
            (19.20, 71.60),  # Mumbai High South
            (19.00, 72.10),  # Traffic Separation Scheme
            (18.92, 72.65),  # Prongs Reef Outer
            (18.96, 72.85),  # Mumbai Inner Harbour / BPCL
        ],
        'start_time': '2026-03-01T08:00:00Z',
        'total_pings': 100,
    },
    {
        'mmsi': 419000345,
        'name': 'ICGS VIKRAM',
        'type': 50,
        'type_str': 'Offshore Patrol Vessel (Coast Guard)',
        'length': 105, 'width': 13.6, 'draft': 3.8, 'cargo': 50,
        'base_speed': 21.5,
        'waypoints': [
            (18.90, 72.70),  # Mumbai Base
            (18.50, 71.50),  # EEZ Grid Alpha
            (17.80, 71.20),  # EEZ Grid Bravo
            (18.20, 70.80),  # Western EEZ Boundary
            (19.00, 71.10),  # Sector 4 Intercept Track
            (19.25, 72.30),  # Mumbai North Approach
        ],
        'start_time': '2026-03-01T02:00:00Z',
        'total_pings': 150,
    }
]
print(f'Configured {len(VESSEL_CORRIDORS)} Indian maritime vessel routes.')

## 3. Kinematic Interpolation & Noise Modeling Engine

In [ ]:
def calculate_bearing(lat1, lon1, lat2, lon2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dlam = math.radians(lon2 - lon1)
    y = math.sin(dlam) * math.cos(phi2)
    x = math.cos(phi1) * math.sin(phi2) - math.sin(phi1) * math.cos(phi2) * math.cos(dlam)
    bearing = (math.degrees(math.atan2(y, x)) + 360) % 360
    return bearing

def generate_interpolated_track(waypoints, total_pings, base_speed, start_time_str):
    start_dt = datetime.fromisoformat(start_time_str.replace('Z', '+00:00'))
    segments = []
    total_dist = 0.0
    for i in range(len(waypoints) - 1):
        lat1, lon1 = waypoints[i]
        lat2, lon2 = waypoints[i+1]
        dist = math.hypot(lat2 - lat1, lon2 - lon1)
        segments.append((lat1, lon1, lat2, lon2, dist))
        total_dist += dist

    rows = []
    current_time = start_dt
    for ping_idx in range(total_pings):
        progress = ping_idx / max(1, total_pings - 1)
        target_dist = progress * total_dist
        accum_dist = 0.0
        curr_lat, curr_lon = waypoints[0]
        curr_cog = 0.0
        for lat1, lon1, lat2, lon2, s_dist in segments:
            if accum_dist + s_dist >= target_dist or s_dist == 0:
                seg_prog = (target_dist - accum_dist) / max(1e-6, s_dist)
                curr_lat = lat1 + (lat2 - lat1) * seg_prog
                curr_lon = lon1 + (lon2 - lon1) * seg_prog
                curr_cog = calculate_bearing(lat1, lon1, lat2, lon2)
                break
            accum_dist += s_dist
            
        lat_noise = random.gauss(0, 0.0008)
        lon_noise = random.gauss(0, 0.0008)
        speed_noise = random.gauss(0, 0.3)
        cog_noise = random.gauss(0, 1.5)
        
        lat = round(curr_lat + lat_noise, 5)
        lon = round(curr_lon + lon_noise, 5)
        sog = round(max(0.5, min(28.0, base_speed + speed_noise)), 1)
        cog = round((curr_cog + cog_noise + 360) % 360, 1)
        
        time_step_sec = random.randint(700, 950)
        current_time += timedelta(seconds=time_step_sec)
        
        rows.append({
            'BaseDateTime': current_time.strftime('%Y-%m-%dT%H:%M:%S'),
            'LAT': lat, 'LON': lon, 'SOG': sog, 'COG': cog, 'Heading': int(cog)
        })
    return rows

print('Trajectory engine initialized successfully.')

## 4. Generate & Save Dataset

In [ ]:
all_rows = []
for v in VESSEL_CORRIDORS:
    pings = generate_interpolated_track(v['waypoints'], v['total_pings'], v['base_speed'], v['start_time'])
    for p in pings:
        all_rows.append({
            'MMSI': v['mmsi'],
            'BaseDateTime': p['BaseDateTime'],
            'LAT': p['LAT'],
            'LON': p['LON'],
            'SOG': p['SOG'],
            'COG': p['COG'],
            'Heading': p['Heading'],
            'VesselName': v['name'],
            'IMO': f'IMO{random.randint(9000000, 9999999)}',
            'CallSign': f'VT{random.randint(1000, 9999)}',
            'VesselType': v['type'],
            'Status': 0,
            'Length': v['length'],
            'Width': v['width'],
            'Draft': v['draft'],
            'Cargo': v['cargo'],
            'TransceiverClass': 'A'
        })

df_india = pd.DataFrame(all_rows)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_india.to_csv(OUT_CSV, index=False)
print(f'✅ Successfully exported {len(df_india):,} AIS rows to {OUT_CSV}')
df_india.head()

## 5. Visualise Vessel Paths Around India Coastline

In [ ]:
plt.figure(figsize=(10, 10), facecolor='#0b1724')
ax = plt.axes()
ax.set_facecolor('#0b1724')

colors = ['#00f2fe', '#b877ff', '#f983e9', '#f59e0b', '#22c55e', '#ef4444']
for idx, mmsi in enumerate(df_india['MMSI'].unique()):
    sub = df_india[df_india['MMSI'] == mmsi]
    name = sub['VesselName'].iloc[0]
    c = colors[idx % len(colors)]
    ax.plot(sub['LON'], sub['LAT'], marker='o', markersize=3, label=name, color=c, linewidth=2)

ax.set_title('Simulated Indian Coastal Maritime Corridors', color='#c6f1f7', fontsize=14)
ax.set_xlabel('Longitude (°E)', color='#7a9bc0')
ax.set_ylabel('Latitude (°N)', color='#7a9bc0')
ax.tick_params(colors='#7a9bc0')
for spine in ax.spines.values():
    spine.set_color('#1e3352')
ax.legend(facecolor='#0b1724', labelcolor='white', loc='upper right', fontsize=8)
plt.grid(True, color='white', alpha=0.1)
plt.show()